<h1>🤖 Getting Started with Agent Tools</h1>

Welcome to this interactive tutorial on building powerful AI agents! In this notebook, we'll explore how to give your agents new abilities using **tools**. 

Tools are what transform a standard Large Language Model (LLM) into a capable agent that can interact with the world, perform calculations, and even ask for your permission before taking action.

**In this notebook, you will learn how to:**

1.  **Create Custom Tools**: Turn any Python function into a tool for your agent.
2.  **Delegate to Other Agents**: Build a specialized agent and use it as a tool within another agent.
3.  **Implement Human-in-the-Loop**: Create workflows that pause to ask for human approval before continuing.
4.  **Use Built-in Tools**: Equip your agent with powerful, ready-to-use tools like Google Search.


## ⚙️ Step 1: Setup

First, let's set up our environment. This involves installing the `google-adk` library and configuring your Google API key.

**To get your API key:**
1. Go to [Google AI Studio](https://aistudio.google.com/app/api-keys) and create an API key.
2. In this Colab notebook, click the **🔑 Key icon** in the left sidebar to open the Secrets tab.
3. Create a new secret named `GOOGLE_API_KEY` and paste your key into the value field.


In [ ]:
# Install the Agent Development Kit (ADK)
!pip install -q google-adk

import os
from google.colab import userdata

try:
    # Get the API key from Colab secrets
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
    print("✅ API Key configured successfully!")
except Exception as e:
    print(f"🔑 Authentication Error: Please add 'GOOGLE_API_KEY' to your Colab secrets. {e}")


In [ ]:
# Import all the necessary components from the ADK
import uuid
from google.genai import types

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner, Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search, AgentTool, ToolContext
from google.adk.code_executors import BuiltInCodeExecutor
from google.adk.apps.app import App, ResumabilityConfig
from google.adk.tools.function_tool import FunctionTool

print("✅ ADK components imported.")


## 🔧 Part 1: Custom Function Tools

A **Function Tool** is the simplest way to give your agent a new skill. You can turn almost any Python function into a tool that your agent can call.

**Best Practices for Creating Function Tools:**
*   **Clear Docstrings**: The agent reads the function's docstring to understand what the tool does and when to use it.
*   **Type Hints**: Use type hints (e.g., `name: str`) so the agent knows what kind of data to pass to the function.
*   **Structured Returns**: Return a dictionary (e.g., `{"status": "success", ...}`) to give the agent clear, predictable results.


In [ ]:
def get_exchange_rate(base_currency: str, target_currency: str) -> dict:
    """Looks up and returns the exchange rate between two currencies.

    Args:
        base_currency: The currency code you are converting from (e.g., "USD").
        target_currency: The currency code you are converting to (e.g., "EUR").

    Returns:
        A dictionary with the status and exchange rate.
    """
    # In a real app, this would call a live API. Here, we use a simple dictionary.
    rate_database = {
        "USD": {"EUR": 0.93, "JPY": 157.50, "INR": 83.58}
    }

    rate = rate_database.get(base_currency, {}).get(target_currency)

    if rate:
        return {"status": "success", "rate": rate}
    else:
        return {"status": "error", "message": "Currency pair not supported."}

# Let's test our function!
print(f"Testing the function: {get_exchange_rate('USD', 'EUR')}")


In [ ]:
# Create an agent that uses our new tool
currency_agent = LlmAgent(
    model=Gemini(model="gemini-1.5-flash-latest"),
    instruction="""You are a helpful currency assistant.
    Use the `get_exchange_rate` tool to find the exchange rate between two currencies.
    Clearly state the rate you found.""",
    tools=[get_exchange_rate] # Pass the function directly into the tools list
)

# The InMemoryRunner is a simple way to run and chat with your agent
runner = InMemoryRunner(agent=currency_agent)

# Let's run the agent!
print("--- Running agent with a function tool ---")
await runner.run_debug("What is the exchange rate from USD to JPY?")


## 🤝 Part 2: Multi-Agent Delegation (Agent as a Tool)

Sometimes, a task is too complex or specialized for one agent. For example, LLMs can sometimes make mistakes with math. We can solve this by creating a specialized `calculation_agent` that is an expert at running code.

We can then give our main agent this `calculation_agent` as a tool. This is a powerful pattern called **delegation**, where one agent offloads a specific task to another, more specialized agent.

We use the `AgentTool` wrapper to turn any agent into a tool.


In [ ]:
# 1. Create the specialist agent for calculations
calculation_agent = LlmAgent(
    model=Gemini(model="gemini-1.5-flash-latest"),
    instruction="""You are a specialized calculator. Your ONLY job is to take a calculation request
    and translate it into a single block of Python code that prints the result. 
    Do NOT provide any explanation or conversational text. ONLY output code.""",
    # The BuiltInCodeExecutor gives the agent the ability to run Python code.
    code_executor=BuiltInCodeExecutor(),
)

# 2. Create the main agent that will delegate to the calculator
enhanced_agent = LlmAgent(
    model=Gemini(model="gemini-1.5-flash-latest"),
    instruction="""You are a smart currency conversion assistant.
    1. First, use the `get_exchange_rate` tool to find the exchange rate.
    2. Then, you MUST use the `calculation_agent` to calculate the final converted amount.
    3. Finally, state the result clearly to the user.""",
    tools=[
        get_exchange_rate,
        AgentTool(agent=calculation_agent) # Use AgentTool to wrap the specialist agent
    ]
)

# 3. Run the enhanced agent
enhanced_runner = InMemoryRunner(agent=enhanced_agent)
print("--- Running agent that delegates to another agent ---")
await enhanced_runner.run_debug("How much is 500 USD in EUR?")


## ⏸️ Part 3: Human-in-the-Loop (Pausing for Approval)

What if an agent needs to perform a sensitive or costly action, like placing a large order or deleting a file? We wouldn't want it to proceed without our permission.

This is where **Human-in-the-Loop** comes in. We can design tools that **pause** execution and wait for a human to approve or reject the action.

To do this, our tool function needs to accept a special `tool_context: ToolContext` argument. This context allows the tool to call `tool_context.request_confirmation()`, which signals to the ADK that the agent should pause and wait for a response.


In [ ]:
# This is a more advanced example that shows a complete approval workflow.

# --- 1. Define the Tool that can Pause ---
def place_shipping_order(num_containers: int, destination: str, tool_context: ToolContext) -> dict:
    """Places a shipping order. Requires approval if ordering more than 5 containers."""
    LARGE_ORDER_THRESHOLD = 5

    # Scenario 1: Small order, auto-approve and finish immediately.
    if num_containers <= LARGE_ORDER_THRESHOLD:
        return {"status": "approved", "message": f"Order for {num_containers} containers auto-approved."}

    # Scenario 2: Large order, first time seeing it. PAUSE and ask for approval.
    if not tool_context.tool_confirmation:
        tool_context.request_confirmation(
            hint=f"⚠️ Large order: {num_containers} containers. Please approve.",
            payload={"num_containers": num_containers, "destination": destination},
        )
        return {"status": "pending", "message": f"Order for {num_containers} containers requires approval."}

    # Scenario 3: Resuming after approval. Check if the human confirmed.
    if tool_context.tool_confirmation.confirmed:
        return {"status": "approved", "message": f"Human approved order for {num_containers} containers."}
    else:
        return {"status": "rejected", "message": f"Human rejected order for {num_containers} containers."}

# --- 2. Define the Agent and App for Resumability ---
shipping_agent = LlmAgent(
    name="shipping_agent",
    model=Gemini(model="gemini-1.5-flash-latest"),
    instruction="You are a shipping coordinator. Use the place_shipping_order tool. If status is pending, inform the user.",
    tools=[FunctionTool(func=place_shipping_order)],
)

# The App wrapper with ResumabilityConfig is required for pause/resume functionality.
shipping_app = App(
    name="shipping_coordinator",
    root_agent=shipping_agent,
    resumability_config=ResumabilityConfig(is_resumable=True),
)

# --- 3. Define the Workflow Logic to Handle the Pause ---
async def run_shipping_workflow(query: str, human_approves: bool):
    print(f"\n{'='*50}\nUser > {query}\n")
    session_id = f"order_{uuid.uuid4().hex[:8]}"
    session_service = InMemorySessionService()
    await session_service.create_session(app_name="shipping_coordinator", user_id="test_user", session_id=session_id)
    runner = Runner(app=shipping_app, session_service=session_service)

    # First call to the agent
    events = [event async for event in runner.run_async(user_id="test_user", session_id=session_id, new_message=types.Content(role="user", parts=[types.Part(text=query)]))]

    # Check if the agent paused for approval
    approval_info = None
    for event in events:
        if event.content and event.content.parts and event.content.parts[0].function_call and event.content.parts[0].function_call.name == "adk_request_confirmation":
            approval_info = {"approval_id": event.content.parts[0].function_call.id, "invocation_id": event.invocation_id}
            break

    if approval_info:
        print(f"⏸️ Agent paused for approval. Human decision: {'APPROVE' if human_approves else 'REJECT'}")
        # Resume the agent with the human's decision
        approval_response = types.Content(role="user", parts=[types.Part(function_response=types.FunctionResponse(id=approval_info["approval_id"], name="adk_request_confirmation", response={"confirmed": human_approves}))])
        async for event in runner.run_async(user_id="test_user", session_id=session_id, new_message=approval_response, invocation_id=approval_info["invocation_id"]):
            if event.content and event.content.parts[0].text:
                print(f"Agent > {event.content.parts[0].text}")
    else:
        # No approval was needed
        for event in events:
            if event.content and event.content.parts[0].text:
                print(f"Agent > {event.content.parts[0].text}")

# --- 4. Run the Workflow ---
# Demo 1: Small order (should auto-approve)
await run_shipping_workflow("Ship 3 containers to Singapore", human_approves=True)

# Demo 2: Large order (should pause and then be approved by our simulated human)
await run_shipping_workflow("Ship 10 containers to Rotterdam", human_approves=True)

# Demo 3: Large order (should pause and then be rejected)
await run_shipping_workflow("Ship 8 containers to Los Angeles", human_approves=False)


## 🎓 Mini-Capstone: Enhancing the Research Assistant

So far, we've built our own tools. But the ADK also comes with powerful **built-in tools** that are ready to use.

Let's upgrade our Research Assistant from Day 1. Its knowledge is limited to when its model was trained. By giving it the `google_search` tool, we can empower it to find real-time, up-to-date information from the web.


In [ ]:
# Configure the Research Assistant with the google_search tool
research_assistant_agent = LlmAgent(
    model=Gemini(model="gemini-1.5-flash-latest"),
    instruction="""You are a helpful research assistant.
    You MUST use the `google_search` tool to find current, up-to-date information to answer the user's questions.
    Synthesize the search results into a clear and concise answer.""",
    tools=[google_search] # Add the built-in tool to the list
)

# Run the agent with a query that requires recent information
research_runner = InMemoryRunner(agent=research_assistant_agent)

print("--- Running Research Assistant with Google Search ---")
await research_runner.run_debug("What were the key announcements at the most recent Google I/O event?")
